# sessions

> Move reviewed dialogs between Ramabana, Claude Code, and Codex.

In [ ]:
#| default_exp sessions

A Drona session is an Aidialog notebook. Imports convert host archives to that notebook. Exports compile the same notebook for another host.

In [ ]:
#| export
from pathlib import Path
import json

from aidialog.dialog import Dialog
from aidialog.hist import dlg2chat
from aidialog.ipynb import read_ipynb, write_ipynb
from fastcore.script import call_parse
from llmsurgery import ant, oai
from llmsurgery.sess import path_dlg, sess_dlg

from drona.rounds import REVIEW_KEY, capture

## Import a session

Claude and Codex use llmsurgery’s session reader. Ramabana uses its persisted activity archive. All three produce the same review notebook.

In [ ]:
#| export
HOSTS = ('ramabana', 'claude', 'codex')

def import_session(
    host,            # `ramabana`, `claude`, or `codex`
    output,          # review notebook path
    session='latest', # session id, prefix, or `latest`
    cwd=None,        # project directory for Claude or newest-session lookup
    history=None,    # Ramabana history path
    codex_home=None, # optional Codex home
    name=None,       # dialog name
):
    "Import one host session as an Aidialog review notebook."
    if host not in HOSTS: raise ValueError(f'host must be one of {HOSTS}')
    if host == 'ramabana':
        kwargs = {} if history is None else {'history': history}
        return capture(output, session=session, name=name, **kwargs)
    if session == 'latest':
        if host == 'claude':
            path = max(ant.sess_dir(cwd).glob('*.jsonl'), key=lambda p:p.stat().st_mtime)
        else: _, path = oai.project_thread(cwd or '.', codex_home or oai.CODEX_HOME)
        dlg = path_dlg('ant' if host == 'claude' else 'oai', path, name=name, mx=None)
    else: dlg = sess_dlg(session, cwd=cwd, codex_home=codex_home, name=name, mx=None)
    source = dict(dlg.meta.get('llmsurgery') or {})
    source['host'] = host
    review = {'status': 'review', 'source': source}
    dlg.meta[REVIEW_KEY] = review
    dlg.mk_message('# Drona review\n\nEdit this dialog in Leela before exporting it.', idx=0,
                   msg_type='note', skipped=1, meta={REVIEW_KEY: review})
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    write_ipynb(dlg, output)
    return output

The host check is deliberately small and can be run without local Claude or Codex sessions.

In [ ]:
from fastcore.test import test_fail

test_fail(lambda: import_session('other', 'x.ipynb'), contains='host must be one of')

## Compile one dialog for each host

Ramabana consumes a bootstrap prompt because its CLI has no prepared-history argument. Claude receives a resumable native session. Codex receives native Responses API items; llmsurgery does not yet expose a rollout writer.

In [ ]:
#| export
def clean_dialog(source):
    "Prompt messages from a Drona notebook, without review notes."
    dlg = read_ipynb(source)
    if not dlg: raise ValueError(f'Could not read dialog {source}')
    prompts = [m for m in dlg if m.msg_type == 'prompt' and not m.skipped]
    if not prompts: raise ValueError('Dialog contains no prompt turns')
    return Dialog(prompts, name=dlg.name, meta=dlg.meta)

def export_session(
    source,          # Drona review notebook
    host,            # `ramabana`, `claude`, or `codex`
    output=None,     # JSON output for Ramabana or Codex
    cwd=None,        # Claude project directory
):
    "Compile one Drona notebook for a target host."
    if host not in HOSTS: raise ValueError(f'host must be one of {HOSTS}')
    dlg = clean_dialog(source)
    if host == 'claude': return ant.dlg2sess(dlg, cwd=cwd)
    if host == 'codex':
        items = list(oai.dlg2items(dlg))
        if output: Path(output).write_text(json.dumps(items, indent=2))
        return items
    from drona.rounds import bootstrap_prompt
    prompt = bootstrap_prompt(source)
    if output: Path(output).write_text(prompt)
    return prompt

## Understand the common dialog

This synthetic dialog proves the portable part of the design. One prompt, one tool call, one result, and one final reply survive Aidialog and compile to Codex items.

In [ ]:
import tempfile
from aidialog.hist import chat2dlg
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult

tmp = Path(tempfile.mkdtemp())
tool = ToolUse(id='call_1', name='search_code', arguments={'query': 'dialog history'})
msgs = [
    Msg('user', [Text('Find the dialog history contract.')]),
    Msg('assistant', [Text('I will inspect the implementation.'), tool]),
    Msg('tool', [ToolResult(id=tool.id, name=tool.name, text='Chat.hist stores canonical messages.')]),
    Msg('assistant', [Text('Urai uses canonical messages in Chat.hist.')]),
]
dlg = chat2dlg(msgs, 'portable', mx=None)
dlg.meta[REVIEW_KEY] = {'status': 'accepted'}
portable = tmp/'portable.ipynb'
write_ipynb(dlg, portable)
roundtrip = dlg2chat(clean_dialog(portable), plain=True)
assert [m.role for m in roundtrip] == ['user', 'assistant', 'tool', 'assistant']
assert roundtrip[1].content[-1].name == 'search_code'
assert roundtrip[2].content[0].text == 'Chat.hist stores canonical messages.'
roundtrip

Codex export keeps the same call id across the function call and its output.

In [ ]:
items = export_session(portable, 'codex')
call = next(item for item in items if item['type'] == 'function_call')
result = next(item for item in items if item['type'] == 'function_call_output')
assert call['call_id'] == result['call_id']
assert call['name'] == 'search_code'
[(item['type'], item.get('role') or item.get('name')) for item in items]

Claude conversion is tested without writing into the real Claude session directory. `dlg2msgs` is the pure conversion used by `dlg2sess`.

In [ ]:
claude_msgs = ant.dlg2msgs(clean_dialog(portable))
call = next(block for message in claude_msgs for block in message['content']
            if block.get('type') == 'tool_use')
result = next(block for message in claude_msgs for block in message['content']
              if block.get('type') == 'tool_result')
assert call['id'] == result['tool_use_id']
assert call['name'] == 'search_code'
[(message['role'], [block['type'] for block in message['content']]) for message in claude_msgs]

## What can move today

A Ramabana, Claude, or Codex session can become a Drona notebook. That notebook can become a resumable Claude session, a Ramabana bootstrap, or Codex-native items. Creating a resumable Codex rollout remains unavailable until llmsurgery exposes a writer.

## Command line

In [ ]:
#| export
@call_parse
def import_cli(host: str, output: str, session: str='latest', cwd: str=None,
               history: str=None, codex_home: str=None, name: str=None):
    "Import a host session as a review notebook."
    print(import_session(host, output, session, cwd, history, codex_home, name))

@call_parse
def export_cli(source: str, host: str, output: str=None, cwd: str=None):
    "Compile a review notebook for another host."
    result = export_session(source, host, output, cwd)
    if host == 'claude': print(result)
    elif output: print(output)
    else: print(json.dumps(result, indent=2) if host == 'codex' else result)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()